# 06.8 - Training Loops & Validation

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

A training loop orchestrates forward pass, loss, backward pass, and parameter updates over epochs. A validation loop measures performance on held-out data to detect overfitting.

## 2. Why Does This Matter?

Many beginners only watch training loss. Without validation monitoring you cannot tell if the model is generalizing or memorizing.

## 3. Prerequisites

- Units 06.4-06.7 (PyTorch, regularization, DataLoader)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Implement complete training + validation loops
- Diagnose overfitting/underfitting from learning curves
- Track and log metrics properly

## 5. Mental Model

Training is sports practice: train, then scrimmage (validate), then adjust strategy. Never judge a team by practice performance alone.


## 6. Backend + Data


In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

torch.manual_seed(42); np.random.seed(42)
X, y = make_classification(n_samples=2000, n_features=20, n_informative=12, n_redundant=4, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
Xtr = torch.tensor(Xtr, dtype=torch.float32); ytr = torch.tensor(ytr, dtype=torch.long)
Xte = torch.tensor(Xte, dtype=torch.float32); yte = torch.tensor(yte, dtype=torch.long)
print("Synthetic classification data ready.")


## 7. Build Train / Validate Functions

A reusable `train_one_epoch` and a `validate` under `torch.no_grad()`.


In [ ]:
def accuracy(pred, target):
    return (pred.argmax(dim=1) == target).float().mean().item()

def train_one_epoch(model, X, y, criterion, optimizer, batch_size=64):
    model.train()
    idx = torch.randperm(len(X))
    tot_loss = 0.0; tot_acc = 0.0; n = 0
    for i in range(0, len(X), batch_size):
        b_idx = idx[i:i+batch_size]
        bx, by = X[b_idx], y[b_idx]
        optimizer.zero_grad()
        pred = model(bx)
        loss = criterion(pred, by)
        loss.backward()
        optimizer.step()
        tot_loss += loss.item() * len(by)
        tot_acc += accuracy(pred, by) * len(by)
        n += len(by)
    return tot_loss / n, tot_acc / n

@torch.no_grad()
def validate(model, X, y, criterion, batch_size=128):
    model.eval()
    tot_loss = 0.0; tot_acc = 0.0; n = 0
    for i in range(0, len(X), batch_size):
        bx, by = X[i:i+batch_size], y[i:i+batch_size]
        pred = model(bx)
        loss = criterion(pred, by)
        tot_loss += loss.item() * len(by)
        tot_acc += accuracy(pred, by) * len(by)
        n += len(by)
    return tot_loss / n, tot_acc / n

model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 2))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)
print("Train/validate functions defined.")


## 8. Run the Loop and Track History

Log epoch-level train AND validation loss + accuracy.


In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
for epoch in range(30):
    tl, ta = train_one_epoch(model, Xtr, ytr, criterion, optimizer)
    vl, va = validate(model, Xte, yte, criterion)
    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["train_acc"].append(ta)
    history["val_acc"].append(va)
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch+1:2d}: tr_loss={tl:.4f} val_loss={vl:.4f} tr_acc={ta:.3f} val_acc={va:.3f}")
print(f"\nFinal val accuracy: {history['val_acc'][-1]:.3f}")


## 9. Plot Learning Curves

The train/val curves reveal overfitting or underfitting.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"], label="val")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train")
ax[1].plot(history["val_acc"], label="val")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout()
plt.savefig('_tmp_curves.png', dpi=80); plt.close()
gap = history['train_acc'][-1] - history['val_acc'][-1]
print(f"Train-val accuracy gap at end: {gap:.3f}")
print("A widening gap signals overfitting; a large gap already = memorize, not generalize.")


## 10. Detect Overfitting Explicitly

Find the epoch where validation loss stops improving (early-stopping signal).


In [ ]:
best_epoch = int(np.argmin(history["val_loss"]))
print(f"Best validation loss at epoch {best_epoch+1} (val_loss={history['val_loss'][best_epoch]:.4f})")
print(f"Validation loss at final epoch {len(history['val_loss'])}: {history['val_loss'][-1]:.4f}")
print("\nIf the model kept training after the best epoch, it overfit. Early stopping would halt there.")


## 11. Common Mistakes & Debugging

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Val acc << train acc | Overfitting / leakage | Check split | Regularize, fix leakage |
| Val loss rising, train falling | Overfitting | Plot both | Early stopping, regularize |
| Both losses high | Underfitting | Check capacity | Bigger model, less reg |
| Very slow | No GPU / data bottleneck | Check device | GPU, optimize loading |

- Always `model.eval()` + `torch.no_grad()` for validation.
- Track epoch-average loss, not the last batch's.

## 12. Real-World Considerations

- Fraud detection: train acc 99.8% but val plateaus at 94% after epoch 15 → apply early stopping at 18.

## 13. When NOT to Use

- A validation split when data is tiny and you must hold everything out; use cross-validation instead.

## 14. Challenge

Add early stopping that halts when validation loss hasn't improved for `patience` epochs.


In [ ]:
# Challenge: early stopping with patience
def train_with_early(model, X, y, Xv, yv, criterion, optmaker, patience=8, epochs=100):
    opt = optmaker(model.parameters())
    best = float('inf'); best_state = None; counter = 0; stopped = epochs
    for e in range(epochs):
        train_one_epoch(model, X, y, criterion, opt)
        vl, va = validate(model, Xv, yv, criterion)
        if vl < best:
            best = vl; best_state = {k: v.clone() for k, v in model.state_dict().items()}; counter = 0
        else:
            counter += 1
            if counter >= patience:
                stopped = e + 1; break
    model.load_state_dict(best_state)
    return stopped, best

torch.manual_seed(1)
m = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 2))
stopped, best = train_with_early(m, Xtr, ytr, Xte, yte, criterion, lambda p: optim.Adam(p, lr=0.005))
print(f"Early stopping halted at epoch {stopped}, best val loss {best:.4f}")


## 15. Closed-Book Recall

Without looking back:

1. Why call `model.eval()` before validation?
2. What does `@torch.no_grad()` do and why for validation?
3. How do you detect overfitting from train/val curves?
4. Difference between epoch loss and batch loss?

## 16. Teach-Back Questions

Explain to another person:

- The structure of a correct training loop.
- How to read a train/val loss plot.

## 17. Summary

You built train/validate functions, tracked both losses and accuracies, plotted learning curves, and added early stopping.

## 18. Further Experiment

- Add an LR scheduler to the loop.
- Track precision/recall/F1 alongside accuracy.

## 19. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy, scikit-learn, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
